# 03 - LLM chat (no API)

1. Run the prompt cell. It creates `results/llm/prompt_1.txt`, `prompt_2.txt`...
2. Paste each prompt in an LLM chat, copy its answer.
3. Paste all answers into `results/llm/answers.jsonl` (one JSON per line).
4. Run the last two cells.

In [5]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from pathlib import Path

from ner.evaluation import evaluate_ner_relaxed, evaluate_ner_strict
from ner.llm import answers_to_predictions, build_prompt
from ner.utils import load_jsonl, save_responses_to_jsonl

In [7]:
VALIDATION_PATH = '../data/plain/validation.jsonl'
LLM_DIR = Path('../results/llm')
ANSWERS_PATH = LLM_DIR / 'answers.jsonl'
OUTPUT_PATH = LLM_DIR / 'llm_validation.jsonl'

N_DOCS = 10   # how many documents to send to the chat
BATCH = 5     # documents per prompt

LABELS = [
    'PERSON_NAME', 'ID_CARD_NUMBER', 'ADDRESS',
    'REGISTRATION_NUMBER', 'DATE', 'AMOUNT', 'ORGANIZATION'
]

## Step 1: create the prompts

In [8]:
samples = load_jsonl(VALIDATION_PATH, n=N_DOCS)
LLM_DIR.mkdir(parents=True, exist_ok=True)
for i in range(0, len(samples), BATCH):
    path = LLM_DIR / f'prompt_{i // BATCH + 1}.txt'
    path.write_text(build_prompt(samples[i:i + BATCH], LABELS), encoding='utf-8')
    print('created', path)

created ../results/llm/prompt_1.txt
created ../results/llm/prompt_2.txt


## Step 2: paste the chat answers in `results/llm/answers.jsonl`, then run:

In [9]:
predictions = answers_to_predictions(ANSWERS_PATH, samples)
print(len(predictions), 'documents with answers')
save_responses_to_jsonl(predictions, OUTPUT_PATH)

10 documents with answers


In [10]:
strict = evaluate_ner_strict(VALIDATION_PATH, OUTPUT_PATH)
relaxed = evaluate_ner_relaxed(VALIDATION_PATH, OUTPUT_PATH)


STRICT EVALUATION
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 410 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
--------------------------------------------------------------------------------------------
PERSON_NAME                 1.00   1.00   1.00     100       0       0
ID_CARD_NUMBER              1.00   1.00   1.00      40       0       0
ADDRESS                     1.00   1.00   1.00      40       0       0
REGISTRATION_NUMBER         1.00   1.00   1.00      50       0       0
DATE                        1.00   1.00   1.00      80       0       0
AMOUNT                      1.00   1.00   1.00      40       0       0
ORGANIZATION                1.00   1.00   1.00      60       0       0

RELAXED EVALUATION - OCCURRENCES
Micro -> P: 1.00 | R: 1.00 | F1: 1.00 | TP: 410 | FP: 0 | FN: 0

ENTITY TYPE                    P      R     F1      TP      FP      FN
-------------------------------------------------------------------------------------